# v3.0 POSi (biogenic silica) -- diatomgraz unlock

PR #60. nb32 corrected the v3.0 5/6-ceiling diagnosis: the actual baseline
binding parameter is `diatomgraz` (2/15 Cal at PR #57 best config; 6/7 of the
5/6-miss seeds drop diatomgraz). POSi (biogenic silica) is the empirically-
motivated action #2: only diatoms produce bSi, so its absolute magnitude pins
`g_diatom * G0_GRAZE * P_diatom` -- the only Carroll-6 parameter combination
that enters silica production.

**Implementation -- Approach C (steady-state diagnostic, not state extension):**

```
bSi_1 = R_SI_C * (mort_diatom + graze_diatom) / W_SINK
bSi_2 = W_SINK * bSi_1 * h1 / [h2 * (R_SI_DISSOL + W_SINK)]
```

Computed from the integrated state's `state[I_DIATOM]` and the seed's
per-cell `diatomgraz` prediction. Autograd-clean. Avoids extending the
15-tracer state vector.

**Result -- diatomgraz Cal hit rate: 13% (baseline) -> 80% (POSi).** First
laptop-tractable lever that moves diatomgraz at 1500 epochs without breaking
the iron pair (alpfe stays Cal in 4-5/5 seeds; **4/5 Excellent at POSI_W=1.0**
-- highest alpfe Excellent rate in the project).

In [ ]:
import glob
import json
from pathlib import Path

import numpy as np
import pandas as pd

def find_repo_root() -> Path:
    p = Path.cwd().resolve()
    for d in [p, *p.parents]:
        if (d / 'src' / 'darwindiff').is_dir():
            return d
    raise RuntimeError(f'repo root not found from {p}')

ROOT = find_repo_root()
SCRIPTS = ROOT / 'scripts'
print(f'ROOT={ROOT}')

PARAMS = ['alpfe', 'scav_rat', 'Smallgrow', 'Biggrow', 'diatomgraz', 'R_PICPOC']
CARROLL = {'alpfe':0.92831,'scav_rat':6.0250e-7,'Smallgrow':0.66098,
           'Biggrow':0.43148,'diatomgraz':0.83003,'R_PICPOC':0.04245}
CAL_BANDS = ('Cal-grade', 'Excellent')

def load_posi(w):
    pat = str(SCRIPTS / f'run_v3.0_joint_eqpac-natlsubpolar_seed*_posiW{w}.json')
    return [json.load(open(f)) for f in sorted(glob.glob(pat))]

weights = ['0.3', '1.0', '3.0']
data = {w: load_posi(w) for w in weights}
for w, rs in data.items():
    print(f'POSI_W={w}: {len(rs)} JSONs loaded')

## 1. Aggregate per POSI_W vs baseline

Baseline (PR #57 best config, n=15) reference numbers from `nb32`:

- 7/15 at 5+/6 Cal-grade, mean_cal = 3.93
- diatomgraz Cal: **2/15 (13%)**
- alpfe Cal: 13/15 (87%)
- 14 total Excellents across 15 seeds (mostly Biggrow + Smallgrow)

In [ ]:
rows = []
for w, rs in data.items():
    n = len(rs); cal = [r['n_cal_grade'] for r in rs]; exc = [r['n_excellent'] for r in rs]
    band_cal = {p: sum(1 for r in rs if r['params'][p]['joint_band'] in CAL_BANDS) for p in PARAMS}
    band_exc = {p: sum(1 for r in rs if r['params'][p]['joint_band'] == 'Excellent') for p in PARAMS}
    rows.append({
        'POSI_W': w, 'n': n,
        'n_at_5+': sum(1 for c in cal if c >= 5),
        'n_at_4+': sum(1 for c in cal if c >= 4),
        'mean_cal': sum(cal)/n,
        'total_exc': sum(exc),
        'alpfe Cal': band_cal['alpfe'], 'alpfe Exc': band_exc['alpfe'],
        'scav_rat Cal': band_cal['scav_rat'],
        'Smallgrow Cal': band_cal['Smallgrow'], 'Biggrow Cal': band_cal['Biggrow'],
        'diatomgraz Cal': band_cal['diatomgraz'],
        'R_PICPOC Cal': band_cal['R_PICPOC'],
    })
pd.DataFrame(rows).set_index('POSI_W').round(2)

## 2. diatomgraz recovery -- the headline

Per-seed diatomgraz value and band, sorted by POSI_W. Carroll target = 0.83003.
Cal-grade threshold = <= 0.50 off Carroll (0.41 <= recovered <= 1.25).

In [ ]:
dg_rows = []
for w, rs in data.items():
    for r in rs:
        dg = r['params']['diatomgraz']
        dg_rows.append({
            'POSI_W': w, 'seed': r['seed'],
            'recovered': dg['joint_recovered'],
            'off': dg['joint_abs_rel_offset'],
            'band': dg['joint_band'],
        })
pd.DataFrame(dg_rows).round(4)

## 3. The scav_rat trade-off -- parameter conservation persists

Even with diatomgraz unblocked, the 5/6 ceiling persists -- the residual
sink moves to **scav_rat**. Look at per-seed bands:

| Family | Dominant 5/6 miss | Mechanism |
|---|---|---|
| Baseline (PR #57) | **diatomgraz** | Chl1 z-score under-constrains diatom growth |
| POSi (this PR) | **scav_rat** | POSi anchors diatomgraz; iron-pair balance shifts |

We can now choose WHICH 5 of 6 are at Cal by selecting the observation mix.
**6/6 path candidate:** POSi + lighter POC_SUB_W -- next-session experiment.

In [ ]:
scav_rows = []
for w, rs in data.items():
    for r in rs:
        scav_rows.append({
            'POSI_W': w, 'seed': r['seed'],
            'scav_rat': r['params']['scav_rat']['joint_recovered'],
            'scav_off': r['params']['scav_rat']['joint_abs_rel_offset'],
            'scav_band': r['params']['scav_rat']['joint_band'],
            'diatomgraz_band': r['params']['diatomgraz']['joint_band'],
            'n_cal': r['n_cal_grade'], 'n_exc': r['n_excellent'],
        })
pd.DataFrame(scav_rows).round(4)

## 4. Findings + next direction

**POSi works as designed.**
- diatomgraz Cal hit rate: 13% baseline -> **80% at POSI_W in {0.3, 1.0}**
- alpfe stays Cal in 4-5/5 seeds (iron pair preserved, unlike PR #59's PIC/POC abs anchors which got alpfe to 0/5)
- **alpfe Excellent: 4/5 at POSI_W=1.0** -- highest in the project (baseline 2/15 = 13%)
- POSI_W=3.0 too strong (overshoots, kills diatomgraz Cal)
- POSI_W in {0.3, 1.0} is the sweet spot

**But 5/6 ceiling persists -- parameter conservation just shifts the residual sink.**

We have empirical evidence that the observations support ~5 effective
constraints. By choosing the observation mix, we choose WHICH 5 of 6 are
reliably Cal-grade. The 6th is always the residual.

**Next direction (laptop-tractable):** POSi + scav_rat-restoration combo. The
current sweep used `POC_SUB_W=3.0` (v2.8 anchor for scav_rat) alongside POSi --
POSi is dominating and overriding the scav_rat constraint. Lighter `POC_SUB_W`
with `POSI_W` might find a balance point where both diatomgraz AND scav_rat
remain Cal-grade. Or try `GEOTRACES_W` (surface iron) at lighter weight as an
alternative scav_rat anchor.

If a balance point exists, this is the laptop path to 6/6 Cal-grade.
If not, parameter conservation is structurally robust to observation tuning,
and the 6/6 plateau is a real ceiling on this box model.